# 3. Pandas DataFrames

NumPy arrays are fast but anonymous — column 3 is just "column 3," not
"tensile_MPa." Pandas provides the `DataFrame` — a labelled 2-D table that
combines the speed of NumPy with the expressiveness of a spreadsheet, where
every row and column has a name you can refer to. It is the go-to tool for
loading, cleaning, and exploring experimental datasets: almost every dataset
you import in this course (from a CSV file, an instrument export, or a
database query) becomes a DataFrame before you do anything else with it.

**Topics**
1. Creating DataFrames and reading files
2. Selecting, filtering, and sorting
3. Missing data
4. Grouping and aggregation
5. Merging datasets
6. Case study: alloy property database

In [ ]:
import pandas as pd
import numpy as np
print('Pandas version:', pd.__version__)

## 3.1 Creating DataFrames

In [ ]:
# From a dictionary (most common way when entering data manually)
df = pd.DataFrame({
    'sample_id':       ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08'],
    'alloy':           ['304SS', '304SS', '316SS', '316SS', 'Mild', 'Mild', 'ToolS', 'ToolS'],
    'heat_treatment':  ['Annealed', 'Q&T', 'Annealed', 'Q&T', 'Annealed', 'Q&T', 'Annealed', 'Q&T'],
    'tensile_MPa':     [515, 890, 485, 760, 400, 650, 760, 1200],
    'yield_MPa':       [205, 690, 170, 540, 250, 520, 420,  900],
    'elongation_pct':  [ 40,  18,  40,  25,  36,  20,  14,    8],
    'hardness_HV':     [180, 280, 160, 230, 130, 210, 220,  380],
})

print(df)
print('\nShape:', df.shape)
print('Column dtypes:')
print(df.dtypes)

In [ ]:
# Quick overview
df.describe()

### Reading from a file

In practice you will load data from CSV or Excel files:

```python
df = pd.read_csv('data/alloys.csv')
df = pd.read_excel('data/alloys.xlsx', sheet_name='Sheet1')
```

Common options: `sep=';'`, `decimal=','`, `header=0`, `index_col=0`, `na_values=['N/A', '-']`.

## 3.2 Selecting, Filtering and Sorting

In [ ]:
# ── Column selection ─────────────────────────────────────────────────────────
print(df['alloy'])                            # single column → Series
print()
print(df[['alloy', 'tensile_MPa', 'hardness_HV']])  # multiple columns → DataFrame

In [ ]:
# ── Label-based (.loc) and position-based (.iloc) ─────────────────────────────
print(df.loc[0, 'tensile_MPa'])          # row 0, column 'tensile_MPa'
print(df.loc[2:4, ['alloy', 'yield_MPa']])  # rows 2-4, two columns
print()
print(df.iloc[0, 3])                     # row 0, column index 3

In [ ]:
# ── Boolean filtering ────────────────────────────────────────────────────────
# Samples with tensile strength > 700 MPa
high_strength = df[df['tensile_MPa'] > 700]
print('High-strength samples:')
print(high_strength[['sample_id', 'alloy', 'heat_treatment', 'tensile_MPa']])

print()
# Stainless steel samples with elongation > 30%
ductile_ss = df[(df['alloy'].str.contains('SS')) & (df['elongation_pct'] > 30)]
print('Ductile stainless steel:')
print(ductile_ss[['sample_id', 'alloy', 'elongation_pct']])

In [ ]:
# ── Sorting ──────────────────────────────────────────────────────────────────
df_sorted = df.sort_values('tensile_MPa', ascending=False)
print('Sorted by tensile strength (descending):')
print(df_sorted[['sample_id', 'alloy', 'heat_treatment', 'tensile_MPa']].to_string(index=False))

## 3.3 Missing Data

Real datasets almost always have gaps — an instrument glitch, a sample that
never got measured, a typo during manual entry. Pandas represents a missing
value as `NaN` ("Not a Number") rather than leaving a blank or a zero,
specifically so it doesn't get silently mistaken for a real measurement of
zero. Before any statistical analysis (Part III onwards), you must decide
what to do with these gaps — the two simplest strategies, shown below, are
to drop incomplete rows entirely or to fill the gap with a reasonable
estimate (such as the column mean). Neither is free of assumptions: dropping
rows loses data, and filling with the mean quietly assumes the missing value
would have been "typical."

In [ ]:
# Simulate a dataset with missing values
rng = np.random.default_rng(99)
df_missing = df.copy()
# Randomly assign NaN to 4 positions
rows = rng.integers(0, len(df), 4)
cols = rng.choice(['elongation_pct', 'hardness_HV', 'tensile_MPa'], 4)
for r, c in zip(rows, cols):
    df_missing.at[r, c] = np.nan

print('Missing value counts per column:')
print(df_missing.isnull().sum())

print('\nRows with any missing value:')
print(df_missing[df_missing.isnull().any(axis=1)])

In [ ]:
# Strategies:
# 1. Drop rows with any missing value
df_dropped = df_missing.dropna()
print('Rows after dropna():', len(df_dropped), '(original:', len(df_missing), ')')

# 2. Fill with column mean (only numeric columns)
df_filled = df_missing.copy()
numeric_cols = df_filled.select_dtypes(include='number').columns
df_filled[numeric_cols] = df_filled[numeric_cols].fillna(df_filled[numeric_cols].mean())

print('After mean-fill, missing count:', df_filled.isnull().sum().sum())

## 3.4 Grouping and Aggregation

`groupby` answers a question you already ask by instinct in the lab: "what's
the average tensile strength *for each alloy*, rather than one average
lumped across all of them?" It works in three steps — split the table into
groups sharing the same value in a column (e.g. one group per alloy), apply
a calculation to each group separately (e.g. the mean), then combine the
results back into a single table. This "split–apply–combine" pattern is the
single most useful tool in Pandas for comparing categories.

In [ ]:
# ── Group by alloy type ───────────────────────────────────────────────────────
grouped = df.groupby('alloy')[['tensile_MPa', 'yield_MPa', 'hardness_HV']]

print('Mean properties per alloy:')
print(grouped.mean().round(0))

print('\nMean properties per alloy × heat treatment:')
print(df.groupby(['alloy', 'heat_treatment'])[['tensile_MPa', 'hardness_HV']].mean().round(0))

In [ ]:
# ── Custom aggregation with .agg() ───────────────────────────────────────────
summary = df.groupby('alloy')['tensile_MPa'].agg(['mean', 'std', 'min', 'max', 'count'])
summary.columns = ['Mean', 'Std', 'Min', 'Max', 'N']
print(summary.round(1))

# ── Adding a derived column ────────────────────────────────────────────────────
df['toughness_approx'] = df['tensile_MPa'] * df['elongation_pct'] / 100  # rough proxy (MPa)
print('\nAppended toughness column:')
print(df[['sample_id', 'tensile_MPa', 'elongation_pct', 'toughness_approx']])

## 3.5 Merging Datasets

In [ ]:
# Corrosion rate data from a separate experiment
df_corrosion = pd.DataFrame({
    'sample_id':        ['A01', 'A03', 'A05', 'A07'],
    'corrosion_mpy':    [0.12, 0.08, 0.45, 0.32],   # mils per year
    'environment':      ['NaCl', 'NaCl', 'NaCl', 'NaCl'],
})

# Merge on 'sample_id' — only annealed samples have corrosion data
df_merged = df.merge(df_corrosion, on='sample_id', how='left')

print('Merged dataset:')
print(df_merged[['sample_id', 'alloy', 'heat_treatment', 'tensile_MPa', 'corrosion_mpy']])
print(f'\n{df_merged["corrosion_mpy"].isna().sum()} samples without corrosion data (NaN)')

## 3.6 Pivoting and Reshaping

In [ ]:
# Pivot table: mean tensile strength by alloy (rows) × heat treatment (columns)
pivot = df.pivot_table(
    values='tensile_MPa',
    index='alloy',
    columns='heat_treatment',
    aggfunc='mean'
)
print('Pivot table — mean tensile strength (MPa):')
print(pivot.to_string())

# Difference between heat treatments
pivot['HT_effect'] = pivot['Q&T'] - pivot['Annealed']
print('\nStrengthening from Q&T (MPa):')
print(pivot[['HT_effect']].sort_values('HT_effect', ascending=False))

---
## Exercises

1. **CSV round-trip**: Save `df` to a CSV file with `df.to_csv('alloys.csv', index=False)`,
   read it back, and verify `df.equals(df_read)` (after resetting data types if needed).

2. **Specific strength**: Add a column `specific_strength` = `tensile_MPa / density_g_cm3`.
   Use the approximate densities: 304SS = 8.0, 316SS = 8.0, Mild = 7.85, ToolS = 7.85.
   (Hint: use `df['alloy'].map(density_dict)`.)

3. **Q-factor**: Compute the index $Q = \text{tensile\_MPa} / \text{hardness\_HV}$.
   For which alloy/heat-treatment combination is $Q$ highest and lowest?